# One-Hot Encoding 实战：心脏病预测中的类别特征编码

**项目目标**：用对照实验展示类别特征编码方式对逻辑回归性能与可解释性的影响。

**核心结论**（详见文末总结）：
- One-hot 优于 label encoding（AUC 0.907 vs 0.891），但小样本上差距不显著
- One-hot 能表达 label encoding 原理上无法表达的非单调类别效应
- `Pipeline` + `ColumnTransformer` 是防数据泄漏的标准工程做法

**数据**：UCI Heart Disease (Cleveland)，297 例（删除 6 例缺失），目标二值化。

## 1. 数据加载

原始文件无表头，手动命名 14 列；`ca`、`thal` 两列的缺失值在原始文件中以 `?` 伪装，
用 `na_values="?"` 翻译成 NaN。

In [1]:
import pandas as pd

COLS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num",
]

df = pd.read_csv(
    "../01.Data/processed.cleveland.data",
    header=None,
    names=COLS,
    na_values="?",
)
print(df.shape)
df.head()

(303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## 2. 特征清点：dtype 是数字 ≠ 变量是数值

全部 13 个特征列的 dtype 都是 float64，但按"对它做算术有没有意义"判定：

| 组 | 列 | 处理方式 |
|---|---|---|
| 连续数值（5） | age, trestbps, chol, thalach, oldpeak | StandardScaler |
| 无序类别（6） | sex, cp, fbs, restecg, exang, thal | OneHotEncoder |
| 序数/两可（2） | slope, ca | 本项目按类别编码（ca 作为计数值也可按数值处理） |

注意 `cp`（胸痛类型）：编码 1~4 中 "4 = 无症状" 临床上反而最危险，
数字顺序不携带严重程度含义，故按无序类别处理。

In [2]:
print(df.nunique())
print("\n缺失值：")
print(df.isna().sum()[df.isna().sum() > 0])

age          41
sex           2
cp            4
trestbps     50
chol        152
fbs           2
restecg       3
thalach      91
exang         2
oldpeak      40
slope         3
ca            4
thal          3
num           5
dtype: int64

缺失值：
ca      4
thal    2
dtype: int64


## 3. 数据准备

- 删除 6 例含缺失值（约 2%，<5% 时删除是合理默认）
- 目标二值化：`num` 0 → 0（无病），1~4 → 1（有病）

In [3]:
df_clean = df.dropna().copy()
df_clean["target"] = (df_clean["num"] > 0).astype(int)

NUM_COLS = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CAT_COLS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

X = df_clean[NUM_COLS + CAT_COLS]
y = df_clean["target"]

print("清洗后形状:", X.shape)
print("目标分布:\n", y.value_counts())

清洗后形状: (297, 13)
目标分布:
 target
0    160
1    137
Name: count, dtype: int64


## 4. 对照实验：三种编码方式

控制变量：同一逻辑回归（`max_iter=5000`）、同一 5 折 CV（shuffle, random_state=42）、
同一指标（ROC AUC）。**唯一变量是编码方式。**

- **实验 A**（错误示范）：类别列当数字直接用（label encoding）
- **实验 B**：`pd.get_dummies` 在 CV 前编码（存在轻微类别清单泄漏）
- **实验 C**（标准做法）：`OneHotEncoder` + `ColumnTransformer` + `Pipeline`，折内编码，零泄漏

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

# 实验 A：label encoding（标准化对全部 13 列，隔离"编码"这一个变量）
pipe_A = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000)),
])
results["A: label encoding"] = cross_val_score(pipe_A, X, y, cv=cv, scoring="roc_auc")

# 实验 B：get_dummies 在 CV 前编码
X_encoded = pd.get_dummies(X, columns=CAT_COLS)
pipe_B = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000)),
])
results["B: get_dummies"] = cross_val_score(pipe_B, X_encoded, y, cv=cv, scoring="roc_auc")

# 实验 C：OneHotEncoder + Pipeline（喂原始未编码 X）
preprocess = ColumnTransformer([
    ("num", StandardScaler(), NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
])
pipe_C = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=5000)),
])
results["C: OneHotEncoder+Pipeline"] = cross_val_score(pipe_C, X, y, cv=cv, scoring="roc_auc")

print(f"{'实验':<28}{'5 折平均 AUC':>14}{'标准差':>10}")
print("-" * 52)
for name, scores in results.items():
    print(f"{name:<28}{scores.mean():>14.4f}{scores.std():>10.4f}")

实验                               5 折平均 AUC       标准差
----------------------------------------------------
A: label encoding                   0.8910    0.0404
B: get_dummies                      0.9023    0.0320
C: OneHotEncoder+Pipeline           0.9067    0.0269


### 结果解读（诚实版）

1. **B > A**（+0.011）：方向与理论一致——one-hot 不给无序类别强加顺序。
   但差距小于折间标准差（0.03~0.04），在 297 例小样本上不显著。
   类别数最多只有 4，逻辑回归能部分补偿强加的顺序，所以 A 没有输得很惨。
2. **B ≈ C**（差 0.004，噪声内）：两者数学上是同一种编码。
   C 的优势在工程：零泄漏、`handle_unknown="ignore"` 兜底、可整体部署。

## 5. 系数解读：one-hot 如何让模型"说实话"

拟合最终 Pipeline，提取 one-hot 后每个类别的系数。

In [5]:
pipe_C.fit(X, y)

feature_names = pipe_C.named_steps["prep"].get_feature_names_out()
coefs = pipe_C.named_steps["model"].coef_[0]

coef_table = pd.DataFrame({"feature": feature_names, "coef": coefs})
print(coef_table.sort_values("coef", ascending=False).to_string(index=False))

         feature      coef
     cat__cp_4.0  1.110596
     cat__ca_2.0  0.922500
   cat__thal_7.0  0.826929
    cat__sex_1.0  0.646133
  cat__slope_2.0  0.519552
    num__oldpeak  0.452423
   num__trestbps  0.394372
     cat__ca_1.0  0.345670
  cat__exang_1.0  0.299725
     cat__ca_3.0  0.279526
    cat__fbs_0.0  0.217400
       num__chol  0.196908
     cat__cp_2.0  0.175491
cat__restecg_2.0  0.169205
cat__restecg_1.0  0.080317
  cat__slope_3.0 -0.054585
        num__age -0.132921
    cat__fbs_1.0 -0.226726
cat__restecg_0.0 -0.258848
  cat__exang_0.0 -0.309050
   cat__thal_6.0 -0.329942
    num__thalach -0.410101
  cat__slope_1.0 -0.474293
   cat__thal_3.0 -0.506312
     cat__cp_3.0 -0.598131
    cat__sex_0.0 -0.655458
     cat__cp_1.0 -0.697282
     cat__ca_0.0 -1.557023


### 关键发现：锯齿形的类别效应

胸痛类型四个系数：`cp_1: -0.70 → cp_2: +0.18 → cp_3: -0.60 → cp_4: +1.11`。

- **无症状型（cp_4）比典型心绞痛（cp_1）危险 1.81 个 log-odds**——与临床认识一致
  （无症状心肌缺血易被忽视，预后反而更差）。
- 这个"忽上忽下"的锯齿模式，label encoding 的单系数直线**原理上无法表达**
  （一个系数 = 一条直线 = 强迫类别效应等距、单调）。
- `thal_7`（可逆性缺损，+0.83）高于 `thal_3`（正常，-0.51）——同样符合医学直觉。
- `thalach`（最大心率）系数为负：运动时心率上得去反映心肺储备好，缺血不严重。

## 6. 总结与局限

**核心教训**：
1. 判断类别变量的标准不是"长什么样"，而是"对它做算术有没有意义"。
2. One-hot 的本质：把"模型只能画一条直线"变成"每个类别一个自由旋钮"。
3. 所有需要 fit 的步骤只准看训练数据——`Pipeline` 把这条纪律变成结构保证。
4. 高基数是 one-hot 的天花板：列爆炸 + 每类样本过少 → 过拟合。

**局限（discovery-only）**：单一数据集、297 例小样本、无外部验证队列；
实验 B 存在轻微类别清单泄漏（已在实验设计中注明）；
`slope`/`ca` 按类别编码，`ca` 作为计数值走数值路线亦合理。